# 52 — Phase 1 Bundle A on **Blind-A** (direct submission)

**Refactor of notebook 51**: skip dev evaluation entirely, run BGE-M3 (config 122) directly on the Blind-A dataset, package as a CodaBench-ready zip, get real leaderboard feedback.

**Why this trade**: Blind-A inference is ~100× faster than dev (80 turns vs 8000). On L4 the full Bundle A run is ~30–60 min instead of ~12 hr. Cost: 1 Blind-A slot per submission (cap = 3/week, you used 1 on 2026-05-15 → 2 left).

**Caveat**: Blind-A noise is ±0.05 nDCG@20 at n=80. If 122 lands within ±0.05 of your current leaderboard (composite 0.21 / nDCG@20 ≈ 0.06 from v5-kto), the BGE-M3 signal is ambiguous. In that case, run cell 7 to also submit baseline 112 for apples-to-apples (2nd slot).

**Decision rule**:
| Outcome | Action |
|---|---|
| nDCG@20 ≥ 0.11 (≥+0.05 vs current 0.06) | clearly better — ship; move to Bundle B/C |
| nDCG@20 in 0.06 ± 0.05 | ambiguous — also run baseline 112 for clean comparison |
| nDCG@20 < 0.04 | regression — pivot to Bundle B (Qwen3-4B) or Bundle C (preprocessing) |

**What this notebook ships**: 
- New configs `112-prorank-rerank-blindsetA.yaml` (baseline) + `122-bge-m3-prorank-rerank-blindsetA.yaml` (experiment), both `use_vllm: false`
- Wraps `run_inference_blindset.py` + the existing `scripts/validate_prediction.py:package_zip()` pattern from notebook 41

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth + Drive (BGE-M3 catalog cache survives across runtimes).
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('NO HF_TOKEN — set it in Colab secrets before running cell 5.', e)

from google.colab import drive
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
print('HF_HOME =', os.environ['HF_HOME'])

DRIVE_EMBED_DIR = '/content/drive/MyDrive/recsys2026_embed_cache'
os.makedirs(DRIVE_EMBED_DIR, exist_ok=True)
# IMPORTANT: configs use cache_dir: "../experiments/cache" which resolves to
# /content/recsys2026/experiments/cache (one dir above music-crs-baselines/).
# The symlink MUST be there so DENSE_LOCAL finds the embedded catalog.
REPO_CACHE_DIR = '/content/recsys2026/experiments/cache/dense_local'
os.makedirs(os.path.dirname(REPO_CACHE_DIR), exist_ok=True)
if os.path.islink(REPO_CACHE_DIR) or os.path.exists(REPO_CACHE_DIR):
    !rm -rf {REPO_CACHE_DIR}
!ln -s {DRIVE_EMBED_DIR} {REPO_CACHE_DIR}
print('symlinked', REPO_CACHE_DIR, '->', DRIVE_EMBED_DIR)

In [ ]:
# 4) Install deps (no vLLM — unstable on Colab and not needed at Blind-A scale).
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml bm25s scipy numpy sentence-transformers

In [ ]:
# 5) Embed BGE-M3 catalog (one-time; skips if cached on Drive).
import os
EMBED_CACHE = f"{REPO_CACHE_DIR}/BAAI_bge-m3/bge-m3-metadata/track_embeddings.pkl"
if os.path.exists(EMBED_CACHE):
    print(f'Cache already exists at {EMBED_CACHE} — skipping embed step.')
else:
    print('No cache found — embedding 47K catalog (~30-60 min on L4).')
    !python scripts/embed_catalog.py \
        --model BAAI/bge-m3 \
        --label bge-m3-metadata \
        --batch-size 64
    print('Embed complete.')

In [ ]:
# 6) Run EXPERIMENT (BGE-M3, config 122) inference on Blind-A.
# 80 turns total → ~30-60 min on L4 with batch_size=32.
%cd /content/recsys2026/music-crs-baselines
EXPERIMENT_TID = '122-bge-m3-prorank-rerank-blindsetA'
EXPERIMENT_PRED = f'exp/inference/blindset_A/{EXPERIMENT_TID}.json'
import os
if os.path.exists(EXPERIMENT_PRED):
    print(f'Experiment predictions exist at {EXPERIMENT_PRED}.')
    print('To force re-run: !rm', EXPERIMENT_PRED)
else:
    print('Running BGE-M3 inference on Blind-A (~30-60 min)...')
    !python run_inference_blindset.py --tid {EXPERIMENT_TID} --batch_size 32
%cd /content/recsys2026

In [ ]:
# 7) (OPTIONAL) Run BASELINE (config 112) inference on Blind-A — only if cell 6 result
# is ambiguous (nDCG@20 within ±0.05 of current leaderboard ≈ 0.06).
# This burns a 2nd Blind-A slot; skip unless apples-to-apples comparison needed.
RUN_BASELINE_BLINDSET = False   # set True only if cell-6 result was ambiguous
%cd /content/recsys2026/music-crs-baselines
BASELINE_TID = '112-prorank-rerank-blindsetA'
BASELINE_PRED = f'exp/inference/blindset_A/{BASELINE_TID}.json'
import os
if not RUN_BASELINE_BLINDSET:
    print('Skipping baseline Blind-A inference. Set RUN_BASELINE_BLINDSET=True to run.')
elif os.path.exists(BASELINE_PRED):
    print(f'Baseline predictions exist at {BASELINE_PRED}.')
else:
    print('Running baseline (Qwen3-0.6B metadata-dense) on Blind-A (~30-60 min)...')
    !python run_inference_blindset.py --tid {BASELINE_TID} --batch_size 32
%cd /content/recsys2026

In [ ]:
# 8) Validate + package the experiment as a CodaBench-ready zip.
# Pattern follows notebook 41's cell 7 — uses scripts/validate_prediction.py.
from datetime import date
import sys, os
sys.path.insert(0, '/content/recsys2026/scripts')
from validate_prediction import load_prediction, validate_schema, package_zip

EXPERIMENT_PRED = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/122-bge-m3-prorank-rerank-blindsetA.json'
ZIP_PATH = f'/content/recsys2026/data/submissions/blindset_A_{date.today().isoformat()}_122_bge_m3.zip'
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)

predictions = load_prediction(EXPERIMENT_PRED)
errors = validate_schema(predictions, 'blindA')
if errors:
    print('SCHEMA VALIDATION FAILED:')
    for e in errors[:20]: print(f'  - {e}')
    raise SystemExit('Refusing to package invalid predictions.')

package_zip(EXPERIMENT_PRED, ZIP_PATH)
print(f'\nReady for CodaBench upload: {ZIP_PATH}')
print(f'  size: {os.path.getsize(ZIP_PATH):,} bytes, {len(predictions)} predictions')

# Also copy to Drive so you can download from there if Colab session ends.
DRIVE_SUBS = '/content/drive/MyDrive/recsys2026_submissions'
os.makedirs(DRIVE_SUBS, exist_ok=True)
import shutil
drive_copy = os.path.join(DRIVE_SUBS, os.path.basename(ZIP_PATH))
shutil.copy(ZIP_PATH, drive_copy)
print(f'  also at: {drive_copy}')

In [ ]:
# 9) (OPTIONAL) Package the baseline submission too — only if you ran cell 7.
if RUN_BASELINE_BLINDSET:
    BASELINE_PRED_ABS = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/112-prorank-rerank-blindsetA.json'
    BASELINE_ZIP = f'/content/recsys2026/data/submissions/blindset_A_{date.today().isoformat()}_112_baseline.zip'
    predictions_b = load_prediction(BASELINE_PRED_ABS)
    errors = validate_schema(predictions_b, 'blindA')
    if errors:
        print('BASELINE SCHEMA FAILED:')
        for e in errors[:20]: print(f'  - {e}')
    else:
        package_zip(BASELINE_PRED_ABS, BASELINE_ZIP)
        print(f'baseline zip: {BASELINE_ZIP}')
        shutil.copy(BASELINE_ZIP, os.path.join(DRIVE_SUBS, os.path.basename(BASELINE_ZIP)))
else:
    print('RUN_BASELINE_BLINDSET=False — no baseline package to make.')

## Submission instructions

1. **Download the zip** from `/content/drive/MyDrive/recsys2026_submissions/` (Drive auto-syncs)
2. Open **CodaBench** → the RecSys 2026 Music CRS challenge → **Blind-A** phase → **Submit**
3. Upload the `.zip` from step 1 (it contains `prediction.json` at the root, per `project_codabench_submission.md`)
4. Wait ~5 min for the leaderboard to update with your scores
5. **Compare** the new `nDCG@20` to your current leaderboard (composite 0.21 / nDCG@20 ≈ 0.06 from the v5-kto submission)
6. **Decision**:
   - **+≥0.05 nDCG@20**: BGE-M3 clearly helps. Move to Bundle B (Qwen3-4B) and stack/replace.
   - **±0.05**: ambiguous — re-run cell 7 with `RUN_BASELINE_BLINDSET=True` for clean baseline
   - **−≥0.05**: pivot to Bundle B / C

**Slots used after this notebook**: 1 if cell 6+8 only, 2 if cell 7+9 also run.